In [ ]:
"""Kaggle GPU translation over the HF corpus: full descriptions to English, 2x Tesla T4.

Reads untranslated rows from hf:// shards of the public corpus dataset,
detects language first, translates titles and descriptions on the GPU, and
uploads the results as a translations OVERLAY keyed by (source_code,
source_job_id). The corpus shards are never rewritten.

Ordering is the whole point: detect -> English check -> translate title and
description. Nothing gates on a field this run has not produced yet, which
is what left 78,011 rows untranslated in the previous Neon-based kernel.
"""
import json, sys, time, pathlib, importlib, traceback
import torch
import torch.multiprocessing as mp

WORK = pathlib.Path("/kaggle/working")
WORK.mkdir(parents=True, exist_ok=True)

CORPUS_REPO_ID = "Shifat2110724169/job-market-intel-corpus"
OVERLAY_REPO_ID = "Shifat2110724169/job-market-intel-translations"
SHARD_PREFIX = "shards/"
ROWS_PER_FETCH = 600
TIME_CAP_S = 3000  # 50 min max budget to preserve weekly GPU quota  # 50 min max budget to preserve weekly GPU quota  # 50 min max budget to preserve weekly GPU quota  # 50 min max budget to preserve weekly GPU quota

state = {"rows_scanned": 0, "rows_translated": 0, "english_copied": 0,
         "detected": 0, "chunks": 0, "chars": 0, "batches": 0,
         "gpu_s": 0.0, "errors": [], "skipped_examples": [],
         "started": time.time()}

# Rows whose translation failed the English-output guard would otherwise be
# re-selected forever, because "description_en IS NULL" is the predicate. v4
# spun on one bad row for hours. Remember what we attempted and treat "nothing
# new" as drained.
attempted: set = set()


def report(stage, extra=None):
    payload = {"stage": stage, "elapsed_s": round(time.time() - state["started"], 1),
               **{k: v for k, v in state.items() if k != "errors"},
               "errors": state["errors"][-5:], **(extra or {})}
    (WORK / "result.json").write_text(json.dumps(payload, indent=2, default=str))
    return payload


def log(msg):
    print(f"[{time.time() - state['started']:8.0f}s] {msg}", flush=True)


# langdetect is a project dependency but NOT in Kaggle's base image. Install it
# once, tolerating an already-present copy (pip exits non-zero on
# "Requirement already satisfied" only with -q; use --no-input and a find_spec
# probe so the common case costs nothing).
import importlib.util as _ilu
_missing = [name for name in ("langdetect", "duckdb", "huggingface_hub", "pyarrow")
            if _ilu.find_spec(name) is None]
if _missing:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing],
                   check=True)
    # Verify, so a silent partial install fails here with a clear message
    # rather than 20 minutes later inside the translation loop.
    still = [n for n in _missing if _ilu.find_spec(n) is None]
    if still:
        raise SystemExit("could not install: " + ", ".join(still))


# --- workers must be real importable modules: papermill has no __main__ file,
# --- and multiprocessing's spawn cannot unpickle a function defined in one.
(WORK / "jmi_worker.py").write_text("\"\"\"Translation worker: one process per GPU. Imported from disk by spawn.\"\"\"\nimport re, time, traceback\nimport torch\n\nMODEL = \"facebook/nllb-200-distilled-600M\"\nCHUNK = 800\nMAX_NEW = 384\nBATCH = 64\n# LANG -> FLORES, mirroring normalization/translation.py\nFLORES = {\n    'af': 'afr_Latn', 'am': 'amh_Ethi', 'ar': 'arb_Arab', 'az': 'azj_Latn',\n    'bg': 'bul_Cyrl', 'bn': 'ben_Beng', 'bs': 'bos_Latn', 'ca': 'cat_Latn',\n    'cs': 'ces_Latn', 'cy': 'cym_Latn', 'da': 'dan_Latn', 'de': 'deu_Latn',\n    'el': 'ell_Grek', 'es': 'spa_Latn', 'et': 'est_Latn', 'eu': 'eus_Latn',\n    'fa': 'pes_Arab', 'fi': 'fin_Latn', 'fil': 'tgl_Latn', 'fr': 'fra_Latn',\n    'ga': 'gle_Latn', 'gl': 'glg_Latn', 'gu': 'guj_Gujr', 'ha': 'hau_Latn',\n    'he': 'heb_Hebr', 'hi': 'hin_Deva', 'hr': 'hrv_Latn', 'hu': 'hun_Latn',\n    'hy': 'hye_Armn', 'id': 'ind_Latn', 'ig': 'ibo_Latn', 'is': 'isl_Latn',\n    'it': 'ita_Latn', 'ja': 'jpn_Jpan', 'ka': 'kat_Geor', 'kk': 'kaz_Cyrl',\n    'km': 'khm_Khmr', 'kn': 'kan_Knda', 'ko': 'kor_Hang', 'ky': 'kir_Cyrl',\n    'lb': 'ltz_Latn', 'lo': 'lao_Laoo', 'lt': 'lit_Latn', 'lv': 'lvs_Latn',\n    'mk': 'mkd_Cyrl', 'ml': 'mal_Mlym', 'mn': 'khk_Cyrl', 'ms': 'zsm_Latn',\n    'mt': 'mlt_Latn', 'my': 'mya_Mymr', 'ne': 'npi_Deva', 'nl': 'nld_Latn',\n    'no': 'nob_Latn', 'pa': 'pan_Guru', 'pl': 'pol_Latn', 'pt': 'por_Latn',\n    'ro': 'ron_Latn', 'ru': 'rus_Cyrl', 'si': 'sin_Sinh', 'sk': 'slk_Latn',\n    'sl': 'slv_Latn', 'so': 'som_Latn', 'sq': 'als_Latn', 'sr': 'srp_Cyrl',\n    'su': 'sun_Latn', 'sv': 'swe_Latn', 'sw': 'swh_Latn', 'ta': 'tam_Taml',\n    'te': 'tel_Telu', 'th': 'tha_Thai', 'tl': 'tgl_Latn', 'tr': 'tur_Latn',\n    'uk': 'ukr_Cyrl', 'ur': 'urd_Arab', 'uz': 'uzn_Latn', 'vi': 'vie_Latn',\n    'yo': 'yor_Latn', 'zh': 'zho_Hans', 'zh-cn': 'zho_Hans', 'zh-tw': 'zho_Hant',\n}\n\n\n# Emoji made NLLB emit 385 whitespace tokens instead of a translation (measured\n# on job 21315: an emoji-heavy ALL-CAPS Vietnamese posting produced an all-blank\n# output, which the guard then skipped forever). Stripping the pictographs fixes\n# it -- cleaned text + beams=1 returned proper English at 5x the speed -- and\n# emoji carry no translatable content, so nothing is lost.\nEMOJI_RE = re.compile(\n    \"[\\\\U0001F000-\\\\U0001FAFF\\\\U00002600-\\\\U000027BF\\\\U0001F1E6-\\\\U0001F1FF\"\n    \"\\\\U0000FE0F\\\\U00002190-\\\\U000021FF\\\\U00002B00-\\\\U00002BFF]+\"\n)\n\n\ndef clean_text(text):\n    text = EMOJI_RE.sub(\" \", text or \"\")\n    return re.sub(r\"\\s{2,}\", \" \", text).strip()\n\n\ndef chunk_text(text):\n    text = clean_text(text)\n    return [text[i:i + CHUNK] for i in range(0, len(text), CHUNK)] or [\"\"]\n\n\ndef load(device_id):\n    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM\n    torch.cuda.set_device(device_id)\n    tok = AutoTokenizer.from_pretrained(MODEL)\n    try:\n        model = AutoModelForSeq2SeqLM.from_pretrained(MODEL, dtype=torch.float16)\n    except TypeError:\n        model = AutoModelForSeq2SeqLM.from_pretrained(MODEL, torch_dtype=torch.float16)\n    return tok, model.to(f\"cuda:{device_id}\").eval()\n\n\ndef translate_shard(device_id, rows, q):\n    \"\"\"rows: [{key, lang, kind, text}] -> q.put({ok, results:[(key,kind,en)], ...})\n\n    kind is \"title\" or \"desc\". Title and description are translated as\n    separate units rather than title-then-description-gated: gating the\n    description on the title is what left 78k rows untranslated last run.\n    \"\"\"\n    try:\n        tok, model = load(device_id)\n        dev = f\"cuda:{device_id}\"\n        eng = tok.convert_tokens_to_ids(\"eng_Latn\")\n\n        by_lang = {}\n        for idx, r in enumerate(rows):\n            flores = FLORES.get((r[\"lang\"] or \"\").lower())\n            if not flores:\n                continue\n            for c in chunk_text(r[\"text\"]):\n                by_lang.setdefault((flores, (r[\"lang\"] or \"\").lower()), []).append((c, idx))\n        if not by_lang:\n            q.put({\"ok\": True, \"results\": [], \"chunks\": 0, \"chars\": 0,\n                   \"elapsed_s\": 0.0, \"rows\": 0})\n            return\n\n        outs = {}\n        first_lang = next(iter(by_lang))\n        tok.src_lang = first_lang[0]\n        enc = tok([by_lang[first_lang][0][0]], return_tensors=\"pt\",\n                  truncation=True).to(dev)\n        with torch.no_grad():                       # warm-up, not timed\n            model.generate(**enc, forced_bos_token_id=eng, num_beams=1, max_new_tokens=8)\n        torch.cuda.synchronize(device_id)\n\n        t0 = time.time()\n        n_chunks = 0\n        for (flores, _lang), items in by_lang.items():\n            items.sort(key=lambda p: len(p[0]))     # length-sorted INSIDE language\n            tok.src_lang = flores\n            for i in range(0, len(items), BATCH):\n                group = items[i:i + BATCH]\n                enc = tok([g[0] for g in group], return_tensors=\"pt\", padding=True,\n                          truncation=True, max_length=512).to(dev)\n                with torch.no_grad():\n                    gen = model.generate(**enc, forced_bos_token_id=eng,\n                                         num_beams=1, max_new_tokens=MAX_NEW)\n                texts = tok.batch_decode(gen, skip_special_tokens=True)\n                if not any(t.strip() for t in texts):\n                    # degenerate batch (all blank): greedy collapsed. One retry\n                    # with beam search usually escapes it; costs nothing when\n                    # it is not needed.\n                    with torch.no_grad():\n                        gen = model.generate(**enc, forced_bos_token_id=eng,\n                                             num_beams=2, max_new_tokens=MAX_NEW)\n                    texts = tok.batch_decode(gen, skip_special_tokens=True)\n                for (_c, idx), text in zip(group, texts):\n                    if text.strip():\n                        outs.setdefault(idx, []).append(text.strip())\n                n_chunks += len(group)\n        torch.cuda.synchronize(device_id)\n        elapsed = time.time() - t0\n\n        results = []\n        for idx, parts in outs.items():\n            r = rows[idx]\n            results.append((r[\"key\"], r[\"kind\"], \" \".join(parts)))\n        q.put({\"ok\": True, \"results\": results, \"chunks\": n_chunks,\n               \"chars\": sum(len(r[\"text\"]) for r in rows),\n               \"elapsed_s\": round(elapsed, 2), \"rows\": len(results)})\n    except Exception:\n        q.put({\"ok\": False, \"error\": traceback.format_exc()[-2000:]})\n")
(WORK / "jmi_detect.py").write_text("\"\"\"Language detection, ported from normalization/translation.py.\"\"\"\nfrom langdetect import DetectorFactory, LangDetectException, detect\n\nDetectorFactory.seed = 0\n\nMIN_DETECTION_CHARS = 10\n\nSCRIPT_RANGES = (\n    (0x3040, 0x30FF, \"ja\"), (0x1100, 0x11FF, \"ko\"), (0xAC00, 0xD7AF, \"ko\"),\n    (0x3400, 0x4DBF, \"zh\"), (0x4E00, 0x9FFF, \"zh\"), (0x0530, 0x058F, \"hy\"),\n    (0xFB13, 0xFB17, \"hy\"), (0x10A0, 0x10FF, \"ka\"), (0x1C90, 0x1CBF, \"ka\"),\n    (0x2D00, 0x2D2F, \"ka\"), (0x1200, 0x137F, \"am\"), (0x1000, 0x109F, \"my\"),\n    (0x1780, 0x17FF, \"km\"), (0x0E80, 0x0EFF, \"lo\"), (0x0D80, 0x0DFF, \"si\"),\n    (0x0600, 0x06FF, \"ar\"),   # Arabic block, incl. Persian (fa extends it)\n)\n\n# Persian-specific letters: U+067E PEH, U+0686 TCHEH, U+0698 JEH, U+06AF GAF.\n# Persian is written in the Arabic block, so a bare range match can only say\n# \"Arabic script\". These four letters appear in almost every Persian word\n# (\"manager\", \"engineer\", \"sales\") and in essentially no MSA text, so their\n# presence disambiguates fa from ar. 60k of the 70k untranslated rows are\n# Persian jobvision postings detected only this way.\nPERSIAN_ONLY = frozenset(\"\\u067e\\u0686\\u0698\\u06af\\u06cc\\u06a9\")\n\n\ndef detect_by_script(text):\n    for lo, hi, code in SCRIPT_RANGES:\n        if any(lo <= ord(ch) <= hi for ch in text):\n            if code == \"ar\" and any(ch in PERSIAN_ONLY for ch in text):\n                return \"fa\"\n            return code\n    return None\n\n\ndef _ascii_ratio(text):\n    letters = [ch for ch in (text or \"\") if ch.isalpha()]\n    return (sum(1 for ch in letters if ord(ch) < 128) / len(letters)) if letters else 0.0\n\n\ndef detect_row(title, description_clean):\n    \"\"\"Detect the posting's language from BOTH fields.\n\n    Checking them separately is unsafe. Measured on the real backlog: a\n    jobvision posting titled \"IT Help Desk\" (English) with a short Persian body\n    probes as English, because choose_detection_text falls back to the title\n    when the description is under 60 chars -- and the row then gets COPIED\n    instead of translated, which is permanent and silent. So any script\n    evidence anywhere in the row outranks the ASCII guard, and the guard only\n    speaks when the whole row is ASCII.\n    \"\"\"\n    combined = f\"{title or ''}\\n{description_clean or ''}\"\n    scripted = detect_by_script(combined)\n    if scripted is not None:\n        return scripted\n    probe = choose_detection_text(title or \"\", description_clean)\n    if _ascii_ratio(combined) >= 0.99:\n        # Fully ASCII in both fields. langdetect is statistically weak here:\n        # it routes \"Senior Backend Engineer\" to German, and a wrong FLORES\n        # source token makes NLLB return garbage rather than an error. So an\n        # all-ASCII row is treated as English unless langdetect is confident\n        # AND the text carries that language's own script. Latin, Cyrillic,\n        # Greek and Arabic families are excluded from the override for\n        # precisely this reason -- \"Barista Part Time\" is English on an\n        # Indonesian board, while Indonesian postings are equally ASCII.\n        try:\n            got = detect(probe).lower()\n        except LangDetectException:\n            return None\n        if got in (\"en\", \"id\", \"ms\", \"tl\", \"sw\", \"so\", \"ht\", \"mg\", \"eo\"):\n            return got\n        return \"en\"\n    if len((probe or \"\").strip()) < MIN_DETECTION_CHARS:\n        return None\n    try:\n        return detect(probe).lower()\n    except LangDetectException:\n        return None\n\n\ndef detect_language(text):\n    \"\"\"ISO 639-1 code for a single string, or None if undecidable.\n\n    Script ranges first: a codepoint match beats a probabilistic profile and\n    survives 3-character titles. Then langdetect above a minimum length.\n    \"\"\"\n    if not text or not text.strip():\n        return None\n    scripted = detect_by_script(text)\n    if scripted is not None:\n        return scripted\n    if len(text.strip()) < MIN_DETECTION_CHARS:\n        return None\n    try:\n        return detect(text).lower()\n    except LangDetectException:\n        return None\n\n\ndef choose_detection_text(title, description_clean):\n    if description_clean and len(description_clean) >= 60:\n        return description_clean\n    return title or description_clean\n")
sys.path.insert(0, str(WORK))
import jmi_worker
import jmi_detect
importlib.reload(jmi_worker)
importlib.reload(jmi_detect)


def open_corpus():
    import duckdb
    return duckdb.connect()


def download_corpus(token):
    """Download the corpus shards once; every query after this is local.

    Reading 826 parquet files through hf:// costs several range requests per
    file -- the footer plus each column chunk -- which is ~5,000 resolver
    requests, exactly the token's 5000-requests/300s budget. Measured:

        RateLimit: "resolvers";r=0;t=30
        RateLimit-Policy: "fixed window";"resolvers";q=5000;w=300

    One full scan can therefore exhaust it by itself. That is why v51 hit HTTP
    429 at offset 14,400 and why v58 died on its first fetch. The whole corpus
    is 0.56 GB, so downloading it costs ~842 requests (one per file) and buys
    fast, unlimited local queries.

    /tmp, not /kaggle/working: the corpus is an input, and anything left in
    working is uploaded as kernel output (0.56 GB of it).
    """
    from huggingface_hub import HfApi, snapshot_download
    api = HfApi(token=token)
    files = api.list_repo_files(CORPUS_REPO_ID, repo_type="dataset")
    wanted = sorted(f for f in files
                    if f.startswith(SHARD_PREFIX) and f.endswith(".parquet"))
    if not wanted:
        raise RuntimeError(
            f"no parquet shards under {SHARD_PREFIX!r} in {CORPUS_REPO_ID}"
        )
    root = pathlib.Path("/tmp/jmi_corpus")
    # Explicit paths rather than "shards/*/*/*.parquet": that glob's meaning
    # depends on the installed huggingface_hub, and the corpus is re-exported
    # underneath us (861 shards when this was written, 123 -- one per source --
    # the next morning, which is also what made v59 die on a 404). A literal
    # list cannot silently select a subset.
    snapshot_download(
        repo_id=CORPUS_REPO_ID,
        repo_type="dataset",
        allow_patterns=wanted,
        token=token,
        local_dir=str(root),
    )
    paths = sorted(str(p) for p in (root / SHARD_PREFIX.rstrip("/")).rglob("*.parquet"))
    if len(paths) < len(wanted):
        # Every missing shard is work that would silently never be selected.
        raise RuntimeError(
            f"downloaded {len(paths)} of {len(wanted)} shards into {root}; "
            "refusing to select work from a partial corpus"
        )
    return paths


def load_overlay_keys(con, token):
    """Load the distinct keys already published, so they are never re-translated.

    The selection predicate used to be "description_en IS NULL" on the corpus,
    but the overlay is a separate dataset: every run restarted at OFFSET 0 and
    re-translated the same first ~14k rows (v51 added 12,404; v53 added 449;
    v56 added 144). Reading the overlay first is what makes a run add rows.

    A failed download is FATAL, not an empty table. Treating the overlay as
    absent would re-translate everything and is the same class of mistake as
    the 2026-10-12 overwrite: publishing without reading what is already there.
    """
    from huggingface_hub import HfApi, hf_hub_download
    api = HfApi(token=token)
    try:
        remote = api.list_repo_files(OVERLAY_REPO_ID, repo_type="dataset")
    except Exception as exc:
        raise RuntimeError(
            f"cannot list {OVERLAY_REPO_ID} ({exc}); refusing to select work "
            "without knowing what is already published"
        ) from exc
    if "translations.parquet" not in remote:
        con.execute("CREATE OR REPLACE TABLE overlay_keys "
                    "(source_code VARCHAR, sid VARCHAR)")
        return 0
    path = pathlib.Path(hf_hub_download(
        repo_id=OVERLAY_REPO_ID,
        filename="translations.parquet",
        repo_type="dataset",
        token=token,
    ))
    con.execute(f"""
        CREATE OR REPLACE TABLE overlay_keys AS
        SELECT DISTINCT source_code, source_job_id::text AS sid
          FROM read_parquet('{path.as_posix()}')
    """)
    n, = con.execute("SELECT count(*) FROM overlay_keys").fetchone()
    return n


def build_pending(con, shards):
    """One row per distinct untranslated key that is not already published.

    Two corrections over row-level selection:

    * the same key appears in several shards (one part file per scrape run), so
      a key translated in one shard but NULL in another was re-selected
      forever. "translated anywhere" is the right test: 47,802 selected keys
      span ~91,600 rows.
    * keys already in the overlay are excluded, which turns a run from a
      re-translation loop into progress on the backlog (12,823 of those 47,802
      keys were already published).

    Fields come from the shard with the longest description, so a fuller
    posting is preferred over a stub.
    """
    files = ", ".join("'" + p.replace("'", "''") + "'" for p in shards)
    con.execute(f"""
        CREATE OR REPLACE TABLE pending AS
        WITH corpus AS (
            SELECT source_code,
                   source_job_id::text AS source_job_id,
                   arg_max(title_original, length(description_clean))    AS title_original,
                   arg_max(title_en, length(description_clean))          AS title_en,
                   arg_max(language_code, length(description_clean))     AS language_code,
                   arg_max(description_clean, length(description_clean)) AS description_clean,
                   max(description_en IS NOT NULL)                       AS desc_translated,
                   max(title_en IS NOT NULL)                             AS title_translated
              FROM read_parquet([{files}], hive_partitioning=true)
             WHERE description_clean IS NOT NULL
             GROUP BY 1, 2
        )
        SELECT source_code, source_job_id, title_original, title_en,
               language_code, description_clean
          FROM corpus c
         WHERE NOT (c.desc_translated AND c.title_translated)
           AND NOT EXISTS (
               SELECT 1 FROM overlay_keys o
                WHERE o.source_code = c.source_code
                  AND o.sid = c.source_job_id
           )
    """)
    n, = con.execute("SELECT count(*) FROM pending").fetchone()
    return n


def fetch_pending(con, offset):
    """A page of the materialised backlog, as plain dicts.

    Paging a local table, not re-scanning 826 remote parquet files. The remote
    scan cost 150-300 s per batch and competed for the resolver quota, which is
    why a 9-hour run only reached ~offset 14,400.
    """
    sql = f"""
        SELECT source_code, source_job_id, title_original, title_en,
               language_code, description_clean
          FROM pending
         ORDER BY md5(source_code || ':' || source_job_id)
         LIMIT {ROWS_PER_FETCH} OFFSET {offset}
    """
    cur = con.execute(sql)
    names = [d[0] for d in cur.description]
    return [dict(zip(names, row)) for row in cur.fetchall()]


def fetch_pending_retry(con, offset, attempts=4):
    """fetch_pending with backoff.

    A transient Hugging Face hiccup (429/5xx) used to end the whole run and
    discard every row already translated. Re-fetching is harmless: the overlay
    merge de-duplicates on the natural key.
    """
    last = None
    for attempt in range(1, attempts + 1):
        try:
            return fetch_pending(con, offset)
        except Exception as exc:
            last = exc
            if attempt == attempts:
                break
            wait = 30 * attempt
            log(f"fetch failed at {offset} (attempt {attempt}/{attempts}): "
                f"{exc}; retrying in {wait}s")
            time.sleep(wait)
    raise last


def looks_non_english(text):
    letters = [ch for ch in text if ch.isalpha()]
    if not letters:
        return False
    return sum(1 for ch in letters if ord(ch) > 0x7F) / len(letters) > 0.5


def run_gpu(rows):
    """Fan the batch across every GPU; returns (results, stats)."""
    n_gpu = max(1, torch.cuda.device_count())
    shards = [rows[i::n_gpu] for i in range(n_gpu) if rows[i::n_gpu]]
    if not shards:
        return [], {"chunks": 0, "chars": 0, "gpu_s": 0.0}
    ctx = mp.get_context("spawn")
    q = ctx.Queue()
    procs = [ctx.Process(target=jmi_worker.translate_shard, args=(i, shards[i], q))
             for i in range(len(shards))]
    for p in procs:
        p.start()
    parts = [q.get() for _ in procs]
    for p in procs:
        p.join()
    results, agg = [], {"chunks": 0, "chars": 0, "gpu_s": 0.0}
    for part in parts:
        if not part.get("ok"):
            state["errors"].append(part.get("error", "unknown"))
            continue
        results.extend(part.get("results", []))
        for k in agg:
            agg[k] += part.get(k, 0)
    return results, agg


def upload_overlay(con, rows):
    """Write the overlay, MERGING with what the repo already holds.

    The previous body said "Additive, never a rewrite" and then did a full
    rewrite: it serialised only this run's rows and uploaded them to the one
    fixed path translations.parquet, so every upload replaced the whole
    dataset. On 2026-10-12 that took the overlay from 24,722 rows to 9,331 and
    silently dropped 24,202 translations while the run still reported success.

    Now the existing file is read back, unioned with this run's rows, and
    de-duplicated on the natural key (source_code, source_job_id) with THIS
    run winning, so a re-translation replaces its stale predecessor.

    If the repository already holds translations.parquet and it cannot be read,
    this RAISES. It does not fall back to publishing this run's rows alone: that
    fallback is the overwrite bug itself. A first attempt used
    HfApi.download_repo_file, which does not exist in huggingface_hub at all, so
    the surrounding try/except swallowed an AttributeError and the kernel
    quietly published 9,331 rows in place of 33,533 -- a silent 24,202-row
    regression that the shrink guard below could not catch, because the failure
    path set prior_rows = 0 before reaching it. Losing a run's work is strictly
    better than destroying earlier runs' work, so the error is fatal.
    """
    import pyarrow as pa
    from huggingface_hub import HfApi, hf_hub_download

    if not rows:
        # Nothing to merge. Serialising an empty list produces a frame with no
        # columns, and DuckDB rejects it with "Provided table/dataframe must
        # have at least one column" -- exactly what v58 reported after its
        # fetch failed. A run with no new rows must be a clean no-op, never a
        # fake upload failure.
        log("no new rows; leaving the published overlay untouched")
        return 0

    token = find_token()
    api = HfApi(token=token)
    WORK.mkdir(parents=True, exist_ok=True)
    target = WORK / "overlay.parquet"

    con.register("new_overlay", pa.Table.from_pylist(rows))

    # Does the repo already hold an overlay? Asked separately from the download
    # so that "no prior file yet" (a legitimate first run) is distinguishable
    # from "the prior file exists but I could not read it" (fatal).
    try:
        remote_files = api.list_repo_files(OVERLAY_REPO_ID, repo_type="dataset")
    except Exception as exc:
        raise RuntimeError(
            f"cannot list files in {OVERLAY_REPO_ID} ({exc}); refusing to upload "
            "blindly, because publishing without reading the prior overlay is "
            "exactly how 24,202 translations were destroyed on 2026-10-12"
        ) from exc
    has_prior = "translations.parquet" in remote_files

    prior_rows = 0
    if has_prior:
        # hf_hub_download, not HfApi.download_repo_file -- the latter does not
        # exist in huggingface_hub and fails with AttributeError.
        downloaded = hf_hub_download(
            repo_id=OVERLAY_REPO_ID,
            filename="translations.parquet",
            repo_type="dataset",
            token=token,
        )
        prior = pathlib.Path(downloaded)
        prior_rows, = con.execute(
            f"SELECT count(*) FROM read_parquet('{prior.as_posix()}')"
        ).fetchone()
        log(f"prior overlay has {prior_rows} rows; merging this run's {len(rows)} in")
    else:
        log(f"{OVERLAY_REPO_ID} has no overlay yet; publishing this run's rows")

    if prior_rows:
        # src_priority and rn must not collide with any real column, and must
        # not collide with EACH OTHER. When both were named priority, DuckDB
        # renamed the second to priority_1 and the filter below silently
        # selected HEAD only -- a merge that looked like it worked.
        #
        # Both bookkeeping columns are EXCLUDED from the output. Publishing
        # src_priority would make the NEXT run fail outright with "UNION ALL BY
        # NAME ... src_priority occurs multiple times", because the column is
        # already present in the file being read back.
        con.execute(f"""
            COPY (
                SELECT * EXCLUDE (rn, src_priority) FROM (
                    SELECT *, row_number() OVER (
                        PARTITION BY source_code, source_job_id
                        ORDER BY src_priority DESC
                    ) AS rn
                    FROM (
                        SELECT *, 0 AS src_priority
                          FROM read_parquet('{prior.as_posix()}')
                        UNION ALL BY NAME
                        SELECT *, 1 AS src_priority FROM new_overlay
                    )
                ) WHERE rn = 1
            ) TO '{target.as_posix()}' (FORMAT PARQUET)
        """)
        expected, = con.execute(f"""
            SELECT count(*) FROM (
                SELECT source_code, source_job_id
                  FROM read_parquet('{prior.as_posix()}')
                UNION
                SELECT source_code, source_job_id FROM new_overlay
            )
        """).fetchone()
    else:
        con.execute(f"COPY new_overlay TO '{target.as_posix()}' (FORMAT PARQUET)")
        expected = len(rows)

    total, = con.execute(
        f"SELECT count(*) FROM '{target.as_posix()}'"
    ).fetchone()

    # The bookkeeping columns must not reach the Hub: src_priority left in the
    # published file makes the NEXT run's "UNION ALL BY NAME" fail with
    # "src_priority occurs multiple times", because the column is already there.
    cols = {r[0] for r in con.execute(f"DESCRIBE SELECT * FROM '{target.as_posix()}'").fetchall()}
    if leaked := {"rn", "src_priority"} & cols:
        raise RuntimeError(
            f"refusing to publish: bookkeeping columns {sorted(leaked)} leaked "
            "into the overlay and would break the next merge"
        )

    if total != expected:
        raise RuntimeError(
            f"overlay merge produced {total} rows but the inputs hold {expected} "
            "distinct natural keys; refusing to publish a lossy overlay"
        )
    if prior_rows and total < prior_rows:
        raise RuntimeError(
            f"refusing to publish {total} rows: the repo already holds {prior_rows}. "
            "A shrinking overlay is the 2026-10-12 regression."
        )

    api.create_repo(OVERLAY_REPO_ID, repo_type="dataset", exist_ok=True)
    api.upload_file(
        path_or_fileobj=str(target),
        path_in_repo="translations.parquet",
        repo_id=OVERLAY_REPO_ID,
        repo_type="dataset",
        token=token,
    )
    log(f"published {total} overlay rows ({len(rows)} new, {prior_rows} prior)")
    return total




#: Secret labels tried in order by find_token. Only one is correct on a given
#: kernel and the label was chosen by hand, so all plausible spellings are
#: attempted. Defined HERE rather than in the script that injects find_token,
#: because the kernel runs standalone: referencing this name without defining
#: it cost a full seven-minute run that ended with
#: "name 'CANDIDATE_LABELS' is not defined" on its final line.
CANDIDATE_LABELS = ("hf_token.txt", "hf_token", "HUGGING_FACE_TOKEN", "huggingface_token")

def find_token():
    # Ask Kaggle for the secret by NAME first. Secrets are served through
    # UserSecretsClient, NOT mounted as readable files under /kaggle, so the
    # old rglob("*.txt") scan found nothing even with the secret correctly
    # attached and ticked -- and its error message claimed otherwise.
    try:
        from kaggle_secrets import UserSecretsClient

        client = UserSecretsClient()
        for label in CANDIDATE_LABELS:
            try:
                token = (client.get_secret(label) or "").strip()
            except Exception:
                continue
            if token.startswith("hf_") and len(token) > 20:
                print(f"[find_token] using Kaggle secret label={label!r}")
                return token
    except ImportError:
        pass

    # Fallback: a secret mounted as a file (older Kaggle behaviour). Scans by
    # content, never by extension, so the label's suffix is irrelevant.
    for p in pathlib.Path("/kaggle").rglob("*"):
        if not p.is_file():
            continue
        try:
            t = p.read_text(errors="ignore").strip()
        except Exception:
            continue
        if t.startswith("hf_") and len(t) > 20:
            print(f"[find_token] using mounted file {p}")
            return t

    raise SystemExit(
        "no HF token found. Tried UserSecretsClient labels "
        f"{CANDIDATE_LABELS} and a /kaggle file scan. Attach the token as a "
        "Kaggle secret and tick its read checkbox."
    )


# ===========================================================================
# HYBRID TITLE CLASSIFICATION (Runs on GPU before translation loop)
# ===========================================================================
CLASSIFY_REPO_ID  = "Shifat2110724169/job-market-intel-classifications"
TAXONOMY_FILENAME = "ref/normalized_job_titles.parquet"
EMBED_MODEL_NAME  = "paraphrase-multilingual-MiniLM-L12-v2"
EMBED_THRESHOLD   = 0.78
CLASSIFY_BATCH_SZ = 1024

_TITLE_NOISE_RE = re.compile(
    r"\s*(?:"
    r"\([^)]*\)"
    r"|\[[^\]]*\]"
    r"|(?:\s*[-|–,]\s*)(?:remote|onsite|hybrid|full-?time|part-?time|urgent|immediate).*$"
    r")$",
    re.IGNORECASE,
)
_SENIORITY_PREFIX_RE = re.compile(
    r"^(?:"
    r"senior|sr\.?|junior|jr\.?|lead|principal|staff|intern|internship"
    r"|stagiaire|associate|head\s+of|director\s+of|vp\s+of|assistant"
    r")\s+",
    re.IGNORECASE,
)
_LITERAL_TEST_RE = re.compile(r"^\s*test\s*\d*\s*$", re.IGNORECASE)
_REMOTEOK_BOILER_RE = re.compile(
    r"Please mention the word \*\*.*?\*\* and tag .*?\. "
    r"This is a beta feature to avoid spam applican\w*\.?",
    re.IGNORECASE | re.DOTALL,
)

def _clean_title(title: str) -> str:
    if not title:
        return ""
    t = title
    for _ in range(3):
        new = _TITLE_NOISE_RE.sub("", t).strip()
        if new == t:
            break
        t = new
    t = re.sub(r"[^\w\s\-\/\&]", "", t)
    return " ".join(t.split())

def _core_title(cleaned: str) -> str:
    return _SENIORITY_PREFIX_RE.sub("", cleaned).strip()

def _strip_boilerplate(desc: str | None) -> str:
    if not desc:
        return ""
    return _REMOTEOK_BOILER_RE.sub("", desc).strip()

def _looks_non_job(title: str, desc: str | None) -> bool:
    if _LITERAL_TEST_RE.match(title):
        return True
    return len(_strip_boilerplate(desc)) < 15

def _build_lookup(taxonomy_rows):
    lookup = {}
    for r in taxonomy_rows:
        key = r["normalized_title"].lower()
        lookup.setdefault(key, r)
        core = _core_title(r["normalized_title"]).lower()
        if core and core != key:
            lookup.setdefault(core, r)
    return lookup

def load_taxonomy(token):
    from huggingface_hub import hf_hub_download
    import pyarrow.parquet as pq

    log(f"downloading taxonomy from {CORPUS_REPO_ID} / {TAXONOMY_FILENAME}")
    path = hf_hub_download(
        repo_id=CORPUS_REPO_ID,
        filename=TAXONOMY_FILENAME,
        repo_type="dataset",
        token=token,
    )
    tbl = pq.read_table(path)
    rows = tbl.to_pydict()
    ids   = rows["normalized_title_id"]
    names = rows["normalized_title"]
    fams  = rows.get("job_family", [None] * len(ids))
    return [
        {"normalized_title_id": i, "normalized_title": n, "job_family": f}
        for i, n, f in zip(ids, names, fams)
    ]

def load_classified_keys(con, token):
    from huggingface_hub import HfApi, hf_hub_download

    api = HfApi(token=token)
    try:
        remote = api.list_repo_files(CLASSIFY_REPO_ID, repo_type="dataset")
    except Exception as exc:
        log(f"classifications repo not found or empty ({exc}); creating table")
        con.execute(
            "CREATE OR REPLACE TABLE classified_keys (source_code VARCHAR, sid VARCHAR)"
        )
        return 0

    if "classifications.parquet" not in remote:
        con.execute(
            "CREATE OR REPLACE TABLE classified_keys (source_code VARCHAR, sid VARCHAR)"
        )
        return 0

    path = pathlib.Path(hf_hub_download(
        repo_id=CLASSIFY_REPO_ID,
        filename="classifications.parquet",
        repo_type="dataset",
        token=token,
    ))
    con.execute(f"""
        CREATE OR REPLACE TABLE classified_keys AS
        SELECT DISTINCT source_code, source_job_id::text AS sid
          FROM read_parquet('{path.as_posix()}')
    """)
    n, = con.execute("SELECT count(*) FROM classified_keys").fetchone()
    return n

def build_classify_pending(con, shards):
    files = ", ".join("'" + p.replace("'", "''") + "'" for p in shards)
    con.execute(f"""
        CREATE OR REPLACE TABLE classify_pending AS
        WITH corpus AS (
            SELECT source_code,
                   source_job_id::text AS source_job_id,
                   arg_max(COALESCE(title_en, title_original), length(description_clean)) AS title_en,
                   arg_max(title_original, length(description_clean))                     AS title_original,
                   arg_max(description_clean, length(description_clean))                  AS description_clean,
                   max(normalized_title_id IS NOT NULL)                                   AS already_classified
              FROM read_parquet([{files}], hive_partitioning=true)
             GROUP BY 1, 2
        )
        SELECT source_code, source_job_id, title_en, title_original, description_clean
          FROM corpus c
         WHERE NOT c.already_classified
           AND NOT EXISTS (
               SELECT 1 FROM classified_keys k
                WHERE k.source_code = c.source_code
                  AND k.sid = c.source_job_id
           )
    """)
    n, = con.execute("SELECT count(*) FROM classify_pending").fetchone()
    return n

def classify_batch_hybrid(rows, taxonomy_rows, lookup, tax_embs, model):
    results = []
    semantic_idxs = []
    semantic_texts = []

    for i, r in enumerate(rows):
        active = (r.get("title_en") or r.get("title_original") or "").strip()
        desc   = r.get("description_clean")

        # Stage 1: deny-list
        if _looks_non_job(active, desc):
            results.append({
                "source_code": r["source_code"],
                "source_job_id": r["source_job_id"],
                "title_classification_status": "excluded_non_job",
                "normalized_title_id": None,
                "normalized_title_confidence": None,
                "title_classified_by": f"hybrid:{EMBED_MODEL_NAME}:v2",
            })
            continue

        cleaned = _clean_title(active)
        core    = _core_title(cleaned)

        # Stage 2a: exact
        hit = lookup.get(cleaned.lower())
        if hit:
            results.append({
                "source_code": r["source_code"],
                "source_job_id": r["source_job_id"],
                "title_classification_status": "matched",
                "normalized_title_id": hit["normalized_title_id"],
                "normalized_title_confidence": 1.00,
                "title_classified_by": f"hybrid:{EMBED_MODEL_NAME}:v2",
            })
            continue

        # Stage 2b: core-exact
        if core and core.lower() != cleaned.lower():
            hit = lookup.get(core.lower())
            if hit:
                results.append({
                    "source_code": r["source_code"],
                    "source_job_id": r["source_job_id"],
                    "title_classification_status": "matched",
                    "normalized_title_id": hit["normalized_title_id"],
                    "normalized_title_confidence": 0.95,
                    "title_classified_by": f"hybrid:{EMBED_MODEL_NAME}:v2",
                })
                continue

        # Stage 3: semantic (deferred)
        semantic_idxs.append(len(results))
        semantic_texts.append(cleaned if cleaned else active)
        results.append({
            "source_code": r["source_code"],
            "source_job_id": r["source_job_id"],
            "title_classification_status": "__pending__",
            "normalized_title_id": None,
            "normalized_title_confidence": None,
            "title_classified_by": f"hybrid:{EMBED_MODEL_NAME}:v2",
        })

    if semantic_texts:
        embs = model.encode(
            semantic_texts,
            batch_size=CLASSIFY_BATCH_SZ,
            normalize_embeddings=True,
            show_progress_bar=False,
            device="cuda:0" if torch.cuda.is_available() else "cpu",
        )
        sims = embs @ tax_embs.T
        best_idxs  = sims.argmax(axis=1)
        best_scores = sims.max(axis=1)

        for slot_i, best_idx, score in zip(semantic_idxs, best_idxs, best_scores):
            score = float(score)
            if score >= EMBED_THRESHOLD:
                results[slot_i]["title_classification_status"]    = "matched"
                results[slot_i]["normalized_title_id"]            = taxonomy_rows[int(best_idx)]["normalized_title_id"]
                results[slot_i]["normalized_title_confidence"]     = round(score, 4)
            else:
                results[slot_i]["title_classification_status"]    = "no_match"
                results[slot_i]["normalized_title_confidence"]     = round(score, 4)

    return results

def upload_classifications_overlay(con, rows, token):
    import pyarrow as pa
    from huggingface_hub import HfApi, hf_hub_download

    if not rows:
        log("no new classifications to upload")
        return 0

    api = HfApi(token=token)
    target = WORK / "classifications.parquet"
    con.register("new_cls", pa.Table.from_pylist(rows))

    try:
        remote_files = api.list_repo_files(CLASSIFY_REPO_ID, repo_type="dataset")
    except Exception as exc:
        log(f"listing {CLASSIFY_REPO_ID} failed ({exc}); will create repo")
        remote_files = []

    has_prior = "classifications.parquet" in remote_files
    prior_rows = 0

    if has_prior:
        downloaded = hf_hub_download(
            repo_id=CLASSIFY_REPO_ID,
            filename="classifications.parquet",
            repo_type="dataset",
            token=token,
        )
        prior = pathlib.Path(downloaded)
        prior_rows, = con.execute(
            f"SELECT count(*) FROM read_parquet('{prior.as_posix()}')"
        ).fetchone()
        log(f"prior classification overlay has {prior_rows} rows; merging {len(rows)} new")
        con.execute(f"""
            COPY (
                SELECT * EXCLUDE (rn, src_priority) FROM (
                    SELECT *, row_number() OVER (
                        PARTITION BY source_code, source_job_id
                        ORDER BY src_priority DESC
                    ) AS rn
                    FROM (
                        SELECT *, 0 AS src_priority
                          FROM read_parquet('{prior.as_posix()}')
                        UNION ALL BY NAME
                        SELECT *, 1 AS src_priority FROM new_cls
                    )
                ) WHERE rn = 1
            ) TO '{target.as_posix()}' (FORMAT PARQUET)
        """)
    else:
        log(f"no prior classification overlay; publishing {len(rows)} rows")
        con.execute(f"COPY new_cls TO '{target.as_posix()}' (FORMAT PARQUET)")

    total, = con.execute(f"SELECT count(*) FROM '{target.as_posix()}'").fetchone()
    if prior_rows and total < prior_rows:
        raise RuntimeError(
            f"refusing to publish {total} rows: repo already holds {prior_rows}."
        )

    api.create_repo(CLASSIFY_REPO_ID, repo_type="dataset", exist_ok=True)
    api.upload_file(
        path_or_fileobj=str(target),
        path_in_repo="classifications.parquet",
        repo_id=CLASSIFY_REPO_ID,
        repo_type="dataset",
        token=token,
    )
    log(f"published {total} classification rows ({len(rows)} new, {prior_rows} prior)")
    return total

def run_classification_phase(con, shards, token):
    log("=== Starting Title Classification Phase (GPU accelerated) ===")
    import subprocess, sys
    import importlib.util as _ilu
    if _ilu.find_spec("sentence_transformers") is None:
        log("installing sentence-transformers...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"],
            check=True,
        )

    from sentence_transformers import SentenceTransformer

    taxonomy_rows = load_taxonomy(token)
    log(f"loaded {len(taxonomy_rows)} taxonomy entries")
    lookup = _build_lookup(taxonomy_rows)
    taxonomy_titles = [r["normalized_title"] for r in taxonomy_rows]

    log(f"loading embedding model {EMBED_MODEL_NAME} onto GPU...")
    model = SentenceTransformer(EMBED_MODEL_NAME, device="cuda:0" if torch.cuda.is_available() else "cpu")
    tax_embs = model.encode(taxonomy_titles, normalize_embeddings=True, show_progress_bar=False, device="cuda:0" if torch.cuda.is_available() else "cpu")
    log(f"taxonomy embeddings computed: {tax_embs.shape}")

    known = load_classified_keys(con, token)
    log(f"already classified overlay keys: {known}")

    backlog = build_classify_pending(con, shards)
    log(f"unclassified jobs backlog: {backlog} rows")
    if backlog == 0:
        log("no unclassified jobs; skipping classification phase")
        return

    all_results = []
    offset = 0
    FETCH_SZ = 5000

    while True:
        cur = con.execute(f"""
            SELECT source_code, source_job_id, title_en, title_original, description_clean
              FROM classify_pending
             ORDER BY md5(source_code || ':' || source_job_id)
             LIMIT {FETCH_SZ} OFFSET {offset}
        """)
        names = [d[0] for d in cur.description]
        batch = [dict(zip(names, row)) for row in cur.fetchall()]
        if not batch:
            break

        offset += len(batch)
        results = classify_batch_hybrid(batch, taxonomy_rows, lookup, tax_embs, model)
        all_results.extend(results)
        log(f"[classify] processed {offset}/{backlog} jobs...")

    try:
        n = upload_classifications_overlay(con, all_results, token)
        log(f"=== Classification Phase Complete: {n} overlay rows published! ===")
    except Exception as exc:
        log(f"CLASSIFICATION UPLOAD ERROR: {exc}")



# ===========================================================================
# HYBRID TITLE CLASSIFICATION (Runs on GPU before translation loop)
# ===========================================================================
import re
CLASSIFY_REPO_ID  = "Shifat2110724169/job-market-intel-classifications"
TAXONOMY_FILENAME = "ref/normalized_job_titles.parquet"
EMBED_MODEL_NAME  = "paraphrase-multilingual-MiniLM-L12-v2"
EMBED_THRESHOLD   = 0.78
CLASSIFY_BATCH_SZ = 1024

_TITLE_NOISE_RE = re.compile(
    r"\s*(?:"
    r"\([^)]*\)"
    r"|\[[^\]]*\]"
    r"|(?:\s*[-|–,]\s*)(?:remote|onsite|hybrid|full-?time|part-?time|urgent|immediate).*$"
    r")$",
    re.IGNORECASE,
)
_SENIORITY_PREFIX_RE = re.compile(
    r"^(?:"
    r"senior|sr\.?|junior|jr\.?|lead|principal|staff|intern|internship"
    r"|stagiaire|associate|head\s+of|director\s+of|vp\s+of|assistant"
    r")\s+",
    re.IGNORECASE,
)
_LITERAL_TEST_RE = re.compile(r"^\s*test\s*\d*\s*$", re.IGNORECASE)
_REMOTEOK_BOILER_RE = re.compile(
    r"Please mention the word \*\*.*?\*\* and tag .*?\. "
    r"This is a beta feature to avoid spam applican\w*\.?",
    re.IGNORECASE | re.DOTALL,
)

def _clean_title(title: str) -> str:
    if not title:
        return ""
    t = title
    for _ in range(3):
        new = _TITLE_NOISE_RE.sub("", t).strip()
        if new == t:
            break
        t = new
    t = re.sub(r"[^\w\s\-\/\&]", "", t)
    return " ".join(t.split())

def _core_title(cleaned: str) -> str:
    return _SENIORITY_PREFIX_RE.sub("", cleaned).strip()

def _strip_boilerplate(desc: str | None) -> str:
    if not desc:
        return ""
    return _REMOTEOK_BOILER_RE.sub("", desc).strip()

def _looks_non_job(title: str, desc: str | None) -> bool:
    if _LITERAL_TEST_RE.match(title):
        return True
    return len(_strip_boilerplate(desc)) < 15

def _build_lookup(taxonomy_rows):
    lookup = {}
    for r in taxonomy_rows:
        key = r["normalized_title"].lower()
        lookup.setdefault(key, r)
        core = _core_title(r["normalized_title"]).lower()
        if core and core != key:
            lookup.setdefault(core, r)
    return lookup

def load_taxonomy(token):
    from huggingface_hub import hf_hub_download
    import pyarrow.parquet as pq

    log(f"downloading taxonomy from {CORPUS_REPO_ID} / {TAXONOMY_FILENAME}")
    path = hf_hub_download(
        repo_id=CORPUS_REPO_ID,
        filename=TAXONOMY_FILENAME,
        repo_type="dataset",
        token=token,
    )
    tbl = pq.read_table(path)
    rows = tbl.to_pydict()
    ids   = rows["normalized_title_id"]
    names = rows["normalized_title"]
    fams  = rows.get("job_family", [None] * len(ids))
    return [
        {"normalized_title_id": i, "normalized_title": n, "job_family": f}
        for i, n, f in zip(ids, names, fams)
    ]

def load_classified_keys(con, token):
    from huggingface_hub import HfApi, hf_hub_download

    api = HfApi(token=token)
    try:
        remote = api.list_repo_files(CLASSIFY_REPO_ID, repo_type="dataset")
    except Exception as exc:
        log(f"classifications repo not found or empty ({exc}); creating table")
        con.execute(
            "CREATE OR REPLACE TABLE classified_keys (source_code VARCHAR, sid VARCHAR)"
        )
        return 0

    if "classifications.parquet" not in remote:
        con.execute(
            "CREATE OR REPLACE TABLE classified_keys (source_code VARCHAR, sid VARCHAR)"
        )
        return 0

    path = pathlib.Path(hf_hub_download(
        repo_id=CLASSIFY_REPO_ID,
        filename="classifications.parquet",
        repo_type="dataset",
        token=token,
    ))
    con.execute(f"""
        CREATE OR REPLACE TABLE classified_keys AS
        SELECT DISTINCT source_code, source_job_id::text AS sid
          FROM read_parquet('{path.as_posix()}')
    """)
    n, = con.execute("SELECT count(*) FROM classified_keys").fetchone()
    return n

def build_classify_pending(con, shards):
    files = ", ".join("'" + p.replace("'", "''") + "'" for p in shards)
    con.execute(f"""
        CREATE OR REPLACE TABLE classify_pending AS
        WITH corpus AS (
            SELECT source_code,
                   source_job_id::text AS source_job_id,
                   arg_max(COALESCE(title_en, title_original), length(description_clean)) AS title_en,
                   arg_max(title_original, length(description_clean))                     AS title_original,
                   arg_max(description_clean, length(description_clean))                  AS description_clean,
                   max(normalized_title_id IS NOT NULL)                                   AS already_classified
              FROM read_parquet([{files}], hive_partitioning=true)
             GROUP BY 1, 2
        )
        SELECT source_code, source_job_id, title_en, title_original, description_clean
          FROM corpus c
         WHERE NOT c.already_classified
           AND NOT EXISTS (
               SELECT 1 FROM classified_keys k
                WHERE k.source_code = c.source_code
                  AND k.sid = c.source_job_id
           )
    """)
    n, = con.execute("SELECT count(*) FROM classify_pending").fetchone()
    return n

def classify_batch_hybrid(rows, taxonomy_rows, lookup, tax_embs, model):
    results = []
    semantic_idxs = []
    semantic_texts = []

    for i, r in enumerate(rows):
        active = (r.get("title_en") or r.get("title_original") or "").strip()
        desc   = r.get("description_clean")

        # Stage 1: deny-list
        if _looks_non_job(active, desc):
            results.append({
                "source_code": r["source_code"],
                "source_job_id": r["source_job_id"],
                "title_classification_status": "excluded_non_job",
                "normalized_title_id": None,
                "normalized_title_confidence": None,
                "title_classified_by": f"hybrid:{EMBED_MODEL_NAME}:v2",
            })
            continue

        cleaned = _clean_title(active)
        core    = _core_title(cleaned)

        # Stage 2a: exact
        hit = lookup.get(cleaned.lower())
        if hit:
            results.append({
                "source_code": r["source_code"],
                "source_job_id": r["source_job_id"],
                "title_classification_status": "matched",
                "normalized_title_id": hit["normalized_title_id"],
                "normalized_title_confidence": 1.00,
                "title_classified_by": f"hybrid:{EMBED_MODEL_NAME}:v2",
            })
            continue

        # Stage 2b: core-exact
        if core and core.lower() != cleaned.lower():
            hit = lookup.get(core.lower())
            if hit:
                results.append({
                    "source_code": r["source_code"],
                    "source_job_id": r["source_job_id"],
                    "title_classification_status": "matched",
                    "normalized_title_id": hit["normalized_title_id"],
                    "normalized_title_confidence": 0.95,
                    "title_classified_by": f"hybrid:{EMBED_MODEL_NAME}:v2",
                })
                continue

        # Stage 3: semantic (deferred)
        semantic_idxs.append(len(results))
        semantic_texts.append(cleaned if cleaned else active)
        results.append({
            "source_code": r["source_code"],
            "source_job_id": r["source_job_id"],
            "title_classification_status": "__pending__",
            "normalized_title_id": None,
            "normalized_title_confidence": None,
            "title_classified_by": f"hybrid:{EMBED_MODEL_NAME}:v2",
        })

    if semantic_texts:
        embs = model.encode(
            semantic_texts,
            batch_size=CLASSIFY_BATCH_SZ,
            normalize_embeddings=True,
            show_progress_bar=False,
            device="cuda:0" if torch.cuda.is_available() else "cpu",
        )
        sims = embs @ tax_embs.T
        best_idxs  = sims.argmax(axis=1)
        best_scores = sims.max(axis=1)

        for slot_i, best_idx, score in zip(semantic_idxs, best_idxs, best_scores):
            score = float(score)
            if score >= EMBED_THRESHOLD:
                results[slot_i]["title_classification_status"]    = "matched"
                results[slot_i]["normalized_title_id"]            = taxonomy_rows[int(best_idx)]["normalized_title_id"]
                results[slot_i]["normalized_title_confidence"]     = round(score, 4)
            else:
                results[slot_i]["title_classification_status"]    = "no_match"
                results[slot_i]["normalized_title_confidence"]     = round(score, 4)

    return results

def upload_classifications_overlay(con, rows, token):
    import pyarrow as pa
    from huggingface_hub import HfApi, hf_hub_download

    if not rows:
        log("no new classifications to upload")
        return 0

    api = HfApi(token=token)
    target = WORK / "classifications.parquet"
    con.register("new_cls", pa.Table.from_pylist(rows))

    try:
        remote_files = api.list_repo_files(CLASSIFY_REPO_ID, repo_type="dataset")
    except Exception as exc:
        log(f"listing {CLASSIFY_REPO_ID} failed ({exc}); will create repo")
        remote_files = []

    has_prior = "classifications.parquet" in remote_files
    prior_rows = 0

    if has_prior:
        downloaded = hf_hub_download(
            repo_id=CLASSIFY_REPO_ID,
            filename="classifications.parquet",
            repo_type="dataset",
            token=token,
        )
        prior = pathlib.Path(downloaded)
        prior_rows, = con.execute(
            f"SELECT count(*) FROM read_parquet('{prior.as_posix()}')"
        ).fetchone()
        log(f"prior classification overlay has {prior_rows} rows; merging {len(rows)} new")
        con.execute(f"""
            COPY (
                SELECT * EXCLUDE (rn, src_priority) FROM (
                    SELECT *, row_number() OVER (
                        PARTITION BY source_code, source_job_id
                        ORDER BY src_priority DESC
                    ) AS rn
                    FROM (
                        SELECT *, 0 AS src_priority
                          FROM read_parquet('{prior.as_posix()}')
                        UNION ALL BY NAME
                        SELECT *, 1 AS src_priority FROM new_cls
                    )
                ) WHERE rn = 1
            ) TO '{target.as_posix()}' (FORMAT PARQUET)
        """)
    else:
        log(f"no prior classification overlay; publishing {len(rows)} rows")
        con.execute(f"COPY new_cls TO '{target.as_posix()}' (FORMAT PARQUET)")

    total, = con.execute(f"SELECT count(*) FROM '{target.as_posix()}'").fetchone()
    if prior_rows and total < prior_rows:
        raise RuntimeError(
            f"refusing to publish {total} rows: repo already holds {prior_rows}."
        )

    api.create_repo(CLASSIFY_REPO_ID, repo_type="dataset", exist_ok=True)
    api.upload_file(
        path_or_fileobj=str(target),
        path_in_repo="classifications.parquet",
        repo_id=CLASSIFY_REPO_ID,
        repo_type="dataset",
        token=token,
    )
    log(f"published {total} classification rows ({len(rows)} new, {prior_rows} prior)")
    return total

def run_classification_phase(con, shards, token):
    log("=== Starting Title Classification Phase (GPU accelerated) ===")
    import subprocess, sys
    import importlib.util as _ilu
    if _ilu.find_spec("sentence_transformers") is None:
        log("installing sentence-transformers...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"],
            check=True,
        )

    from sentence_transformers import SentenceTransformer

    taxonomy_rows = load_taxonomy(token)
    log(f"loaded {len(taxonomy_rows)} taxonomy entries")
    lookup = _build_lookup(taxonomy_rows)
    taxonomy_titles = [r["normalized_title"] for r in taxonomy_rows]

    log(f"loading embedding model {EMBED_MODEL_NAME} onto GPU...")
    model = SentenceTransformer(EMBED_MODEL_NAME, device="cuda:0" if torch.cuda.is_available() else "cpu")
    tax_embs = model.encode(taxonomy_titles, normalize_embeddings=True, show_progress_bar=False, device="cuda:0" if torch.cuda.is_available() else "cpu")
    log(f"taxonomy embeddings computed: {tax_embs.shape}")

    known = load_classified_keys(con, token)
    log(f"already classified overlay keys: {known}")

    backlog = build_classify_pending(con, shards)
    log(f"unclassified jobs backlog: {backlog} rows")
    if backlog == 0:
        log("no unclassified jobs; skipping classification phase")
        return

    all_results = []
    offset = 0
    FETCH_SZ = 5000

    while True:
        cur = con.execute(f"""
            SELECT source_code, source_job_id, title_en, title_original, description_clean
              FROM classify_pending
             ORDER BY md5(source_code || ':' || source_job_id)
             LIMIT {FETCH_SZ} OFFSET {offset}
        """)
        names = [d[0] for d in cur.description]
        batch = [dict(zip(names, row)) for row in cur.fetchall()]
        if not batch:
            break

        offset += len(batch)
        results = classify_batch_hybrid(batch, taxonomy_rows, lookup, tax_embs, model)
        all_results.extend(results)
        log(f"[classify] processed {offset}/{backlog} jobs...")

    try:
        n = upload_classifications_overlay(con, all_results, token)
        log(f"=== Classification Phase Complete: {n} overlay rows published! ===")
    except Exception as exc:
        log(f"CLASSIFICATION UPLOAD ERROR: {exc}")


def main():
    # Resolve the token BEFORE any expensive work. find_token() is otherwise
    # first reached inside upload_overlay(), which runs only at the first
    # checkpoint -- after five batches of translation. A run without a usable
    # secret therefore loaded the model, translated for as long as it could,
    # and then raised SystemExit and discarded every row. Two runs were lost
    # this way, one after 1807s of work. A token failure must cost seconds.
    token = find_token()
    log("token resolved before starting")

    con = open_corpus()
    info = {"torch": torch.__version__, "gpus": torch.cuda.device_count(),
            "corpus": CORPUS_REPO_ID, "overlay": OVERLAY_REPO_ID}
    report("start", info)
    log(json.dumps(info))
    if info["gpus"] == 0:
        # enable_gpu is true in kernel-metadata.json, so 0 here means Kaggle
        # allocated no GPU slot, not that GPU was switched off. torch is
        # resolved before any user code runs, so a +cpu build proves it. A CPU
        # run cannot translate this notebook: the workers call
        # torch.cuda.set_device(), which raises AttributeError on a CPU-only
        # build. Fail now, with a readable message, instead of 55 seconds in.
        report("failed", {
            "error": "no GPU allocated; torch.cuda.device_count() == 0",
            "torch": torch.__version__,
            "cuda": torch.version.cuda,
        })
        raise SystemExit(
            "no GPU allocated: Kaggle gave this session a CPU-only torch build. "
            "Re-run when a GPU slot is available."
        )

    # Smoke test: load the model and translate one short row on each GPU
    # BEFORE the main loop. A tokenizer/spawn/model failure then surfaces in
    # ~30s with a traceback instead of after minutes of fetching, and it
    # proves the NLLB source token actually returns English.
    try:
        smoke = [{"key": "smoke:0", "lang": "fa", "kind": "desc",
                  "text": "این یک آگهی شغلی نمونه است."}]
        smoke_res, smoke_stats = run_gpu(smoke)
        report("smoke", {"smoke_results": smoke_res, "smoke_stats": smoke_stats})
        log(f"smoke ok: {smoke_res}")
    except BaseException:
        import traceback
        tb = traceback.format_exc()
        state["errors"].append(tb[-3000:])
        report("SMOKE_CRASH", {"traceback": tb[-3000:]})
        print(tb)
        raise

    # One download and one full scan. Every fetch after this is local.
    shards = download_corpus(token)
    log(f"corpus downloaded: {len(shards)} shards")
    # Run Title Classification first on GPU (~3-5 mins)
    try:
        run_classification_phase(con, shards, token)
    except Exception as exc:
        log(f"title classification phase failed: {exc}")

    # Run Title Classification first on GPU (~3-5 mins)
    try:
        run_classification_phase(con, shards, token)
    except Exception as exc:
        log(f"title classification phase failed: {exc}")

    known = load_overlay_keys(con, token)
    log(f"overlay already covers {known} keys")
    backlog = build_pending(con, shards)
    log(f"pending backlog: {backlog} distinct untranslated keys")
    if backlog == 0:
        log("nothing to translate: the corpus is fully covered")

    overlay = []
    offset = 0
    while True:
        if time.time() - state["started"] > TIME_CAP_S:
            log("time cap reached")
            break
        try:
            batch = fetch_pending_retry(con, offset)
        except Exception as exc:
            state["errors"].append(f"fetch@{offset}: {exc}")
            log(f"fetch failed at {offset}: {exc}")
            state["fatal"] = True
            break
        if not batch:
            log("corpus drained")
            break
        offset += len(batch)
        state["rows_scanned"] += len(batch)

        gpu_rows = []
        for r in batch:
            key = f"{r['source_code']}:{r['source_job_id']}"
            if key in attempted:
                continue
            attempted.add(key)
            title = r["title_original"] or ""
            desc = r["description_clean"] or ""
            if not title and not desc:
                continue

            # detect_row, not choose_detection_text + detect_language: the
            # title alone can read as English while the body is Persian, and
            # that row would be copied instead of translated.
            lang = r["language_code"]
            if lang in (None, "", "unknown"):
                lang = jmi_detect.detect_row(title, desc)
                state["detected"] += 1
            lang = (lang or "").lower()

            base = {"key": key, "lang": lang, "source_code": r["source_code"],
                    "source_job_id": r["source_job_id"]}
            if lang in ("en", ""):
                # English: copy rather than translate. Correct by definition and
                # free -- 7,023 rows avoided the GPU this way.
                state["english_copied"] += 1
                overlay.append({**base, "language_code": lang or "en",
                                "title_en": r["title_en"] or title or None,
                                "description_en": desc or None,
                                "method": "copy_english"})
                continue
            # Anti-retranslation: only send title to GPU if not already translated!
            if title and not r.get("title_en"):
                gpu_rows.append({**base, "kind": "title", "text": title})
            if desc and not r.get("description_en"):
                gpu_rows.append({**base, "kind": "desc", "text": desc})

        if gpu_rows:
            results, agg = run_gpu(gpu_rows)
            for k, v in agg.items():
                if k == "gpu_s":
                    state["gpu_s"] += v
                else:
                    state[k] += v
            by_key = {}
            for key, kind, text in results:
                slot = by_key.setdefault(key, {"source_code": key.split(":", 1)[0],
                                              "source_job_id": key.split(":", 1)[1]})
                slot["title_en" if kind == "title" else "description_en"] = text
            for key, slot in by_key.items():
                base = next(g for g in gpu_rows if g["key"] == key)
                # Preserve existing title_en if only description was sent to GPU
                orig_r = next((b for b in batch if f"{b['source_code']}:{b['source_job_id']}" == key), None)
                if orig_r and orig_r.get("title_en") and "title_en" not in slot:
                    slot["title_en"] = orig_r["title_en"]
                overlay.append({**slot, "language_code": base["lang"],
                                "method": "nllb"})
                state["rows_translated"] += 1

        state["batches"] += 1
        if state["batches"] % 5 == 0:
            report("progress")
            # Checkpoint after every 5 batches so a kill at hour 8 still leaves
            # usable work on disk rather than nothing.
            try:
                upload_overlay(con, overlay)
            except Exception as exc:
                state["errors"].append(f"checkpoint: {exc}")
                log(f"checkpoint upload failed: {exc}")

    try:
        n = upload_overlay(con, overlay)
        log(f"uploaded {n} overlay rows")
    except Exception as exc:
        state["errors"].append(f"final upload: {exc}")
        log(f"FINAL UPLOAD FAILED: {exc}")
        (WORK / "overlay.parquet").write_bytes(b"")   # leave a marker
    report("done", {"overlay_rows": len(overlay), "offset": offset})
    # A fetch that fails after every retry leaves nothing published. Reporting
    # success here is exactly how the 2026-10 outage stayed invisible for weeks:
    # Kaggle said COMPLETE for runs that translated zero rows.
    if state.get("fatal") and not overlay:
        log("nothing was published; exiting non-zero")
        return 1
    return 0


# v1-v3 all reached KernelWorkerStatus.ERROR with a 0-byte log and result.json
# frozen at "start": only fetch_pending was guarded, so anything raised in the
# row loop or the upload killed the notebook and lost the traceback. Wrap the
# whole thing so a failure is always written where the laptop can read it.
try:
    _rc = main()
except SystemExit:                        # guard already wrote result.json
    raise
except BaseException:                     # noqa: BLE001 - must never lose the log
    import traceback
    tb = traceback.format_exc()
    state["errors"].append(tb[-3000:])
    report("CRASH", {"traceback": tb[-3000:]})
    print(tb)
    _rc = 1
print("EXIT", _rc)
